# Compressed MusicFM: iPhone benchmark and palette-head combination
Mac, kernel SongFormer Python3.11, root Section-Annotations.
Bagian A memakai cache MusicFM yang sudah selesai: encoder tidak dijalankan ulang.
Bagian B membuat proyek benchmark iPhone terpisah, bukan mengubah SongMap.
Default hanya membundel INT8 dan palette6; FP32 besar tetap menjadi referensi Mac.


In [ ]:
import subprocess,sys,json
from pathlib import Path
from datetime import datetime
assert Path('finetuning/config.json').is_file()
comparisons=sorted(Path('outputs.local').glob('musicfm-compression-*/comparison.json'),
    key=lambda p:p.stat().st_mtime,reverse=True)
assert comparisons,'Hasil kompresi MusicFM belum ditemukan'
comparison=comparisons[0]
source=comparison.parent
heads=sorted(Path('outputs.local').glob('coreml-compression-*/palette6-validation.json'),
    key=lambda p:p.stat().st_mtime,reverse=True)
assert heads,'Validasi head palette6 belum ditemukan'
head_validation=heads[0]
head_compression=head_validation.parent/'palette6/compression_report.json'
assert head_compression.is_file()
output=Path('outputs.local')/('musicfm-device-combo-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
output.mkdir(exist_ok=False)
print('MusicFM:',comparison)
print('Palette head:',head_validation)
print('Output:',output)

def run_logged(command,log_path):
    with log_path.open('w') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
            code=process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f'Proses gagal ({code}); lihat {log_path}')


## A. Evaluate cached encoder + palette6 head
Tidak membaca audio atau menjalankan encoder. Seluruh validation menggunakan cache kandidat yang sudah tersedia dan label manual yang sama.

In [ ]:
for scheme in ('int8','palette6'):
    run_logged([sys.executable,'-u','-m','deployment.evaluate_cached_combination',
        '--encoder-pipeline-report',str(source/f'{scheme}-all-validation.json'),
        '--head-validation-report',str(head_validation),
        '--head-compression-report',str(head_compression),
        '--output',str(output/f'{scheme}-palette-head.json')],
        output/f'{scheme}-palette-head.log')


In [ ]:
for scheme in ('int8','palette6'):
    r=json.loads((output/f'{scheme}-palette-head.json').read_text())
    print('MusicFM:',scheme,'| Head:',r['head_scheme'])
    print('F1:',r['metrics']['macro_f1_present'],'| Accuracy:',r['metrics']['section_accuracy'])
    print('Delta vs FP32 head, same encoder:',r['macro_f1_delta_from_fp32_head_same_encoder'])
    print('Per-class:',r['metrics']['per_class'])


## B. Prepare the isolated iPhone encoder benchmark
Dua fixture: prefix750frame (~30s ekuivalen) dari input konvolusi yang sama, dan window Pilihanku lengkap2977frame. Ini mengukur MusicFM saja, bukan frontend/MuQ/head atau seluruh lagu.

In [ ]:
phone=output/'iphone-project'
run_logged([sys.executable,'-u','-m','deployment.prepare_musicfm_iphone',
    '--comparison',str(comparison),'--output-dir',str(phone)],output/'iphone-prepare.log')
subprocess.run(['open',str(phone/'Benchmark.xcodeproj')],check=True)


Di Xcode pilih Signing & Capabilities -> Team Anda dan iPhone17 fisik.
Proyek baru menggunakan paket shape statis terpisah per fixture. Mulai dengan
cpuOnly, int8, short; kemudian failed_full dan palette6. Setelah CPU berjalan,
uji all atau cpuAndNeuralEngine secara eksplisit dan bagikan JSON setiap kasus.

Tutup/buka ulang app antara kandidat untuk perbandingan yang lebih bersih.
Snapshot footprint bukan peak memory. Requested compute units bukan bukti ANE.
Model statis per fixture menduplikasi bobot untuk benchmark, bukan format bundel
produksi final. Paket aslinya tidak dikompresi/ditraining ulang.

Jika memperbarui proyek setelah crash MPSGraph: jalankan ulang setup agar output
baru dibuat, lalu bagian B saja. Bagian A tidak perlu diulang. C++/Metal assertion
menghentikan proses dan tidak dapat dipulihkan dengan Swift do/catch; jangan
mengandalkan automatic fallback setelah assertion. Log Xcode bertanda
MUSICFM_BENCHMARK_START/PREDICT mencatat kasus yang sedang berjalan.
